# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0mneeha93/ML-Track/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

!pip install -q duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print("DuckDB ready.")

DuckDB ready.


## 2. My model under an honest split (before/after)

The out of time test (KMeans fit on February, applied without refitting to March) shows the same four archetypes reappear with closely matching profiles position, impressions, and CTR are all within a similar range for each cluster, and page counts are comparable (the largest shift is the "High Traffic Workhorses" cluster, which grew from 2,776 to 4,020 pages). This is a materially stronger test than the Week 5 stability check, which only resampled rows within the same March window and couldn't have caught drift across time. Observed result: the clustering structure appears directionally stable out of time, not just stable under resampling though one month of out o f time validation is a limited window, and this should be treated as a decision support signal rather than proof of long term stability.

In [2]:
features_feb = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions) ELSE 0 END as ctr,
        c.word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_avg_position > 0
    GROUP BY f.content_hash_id, c.word_count
""").df()

features_feb["word_count"] = features_feb["word_count"].fillna(features_feb["word_count"].median())
print("Feb rows:", len(features_feb))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feb rows: 151956


In [3]:
features_df = con.execute("""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_avg_position) as avg_position,
        SUM(f.gsc_impressions) as impressions_total,
        CASE WHEN SUM(f.gsc_impressions) > 0 THEN SUM(f.gsc_clicks)*1.0/SUM(f.gsc_impressions) ELSE 0 END as ctr,
        c.word_count
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet' f
    JOIN 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet' c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_avg_position > 0
    GROUP BY f.content_hash_id, c.word_count
""").df()

features_df["word_count"] = features_df["word_count"].fillna(features_df["word_count"].median())
print("March rows:", len(features_df))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March rows: 175304


In [4]:
X_feb = features_feb[["avg_position", "impressions_total", "ctr", "word_count"]]
scaler_feb = StandardScaler().fit(X_feb)
X_feb_scaled = scaler_feb.transform(X_feb)

km_feb = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_feb_scaled)
feb_centers = pd.DataFrame(km_feb.cluster_centers_, columns=["avg_position","impressions_total","ctr","word_count"])
print("Feb-fit-on-Feb centers:")
print(feb_centers.round(3))

Feb-fit-on-Feb centers:
   avg_position  impressions_total     ctr  word_count
0        -0.111              0.931  -0.053       1.679
1         2.248             -0.252  -0.085      -0.226
2        -0.327             -0.117  -0.027      -0.246
3        -0.226             -0.291  19.486      -1.067


In [5]:
X_march = features_df[["avg_position", "impressions_total", "ctr", "word_count"]]
X_march_scaled = scaler_feb.transform(X_march)          # reuse Feb's scaler — do NOT refit
march_labels_oot = km_feb.predict(X_march_scaled)        # reuse Feb's cluster model — do NOT refit

features_df["cluster_oot"] = march_labels_oot
oot_profile = features_df.groupby("cluster_oot")[["avg_position","impressions_total","ctr","word_count"]].mean()
oot_profile["n_pages"] = features_df["cluster_oot"].value_counts().sort_index()

print("Feb-fit model applied OUT-OF-TIME to March data:")
print(oot_profile.round(3))

Feb-fit model applied OUT-OF-TIME to March data:
             avg_position  impressions_total    ctr  word_count  n_pages
cluster_oot                                                             
0                  14.031           7161.803  0.003    4191.313    23393
1                  51.125            276.307  0.001     2643.38    29462
2                   9.431            855.552  0.004    2485.221   122157
3                   8.714              1.932  0.727    1423.274      292


## 3. Leakage audit
CTR dominance: In the Week 5 clustering, CTR had by far the largest spread across cluster centers (9.03 vs. 2.90 for impressions), suggesting it was the dominant axis separating clusters. Refitting without CTR did not reduce cluster quality silhouette score actually improved slightly (0.470 with CTR vs. 0.493 without). This is the "too good, investigate don't celebrate" signal from the audit checklist: CTR's outsized influence on the raw center spread looks more like a scaling artifact than a genuinely useful separator.

Unstable low  count CTR artifact: The Week 5 "Suspicious Outliers" cluster (394 pages, ~62–72% CTR) was built almost entirely on pages with 1 to 2 total impressions, where CTR is a near meaningless ratio (1 click / 1 impression = "100%"). Refitting after filtering to pages with at least 20 impressions removed this cluster entirely all four resulting clusters have CTR values in a normal range (max ~3%), with 44,650 low impression rows excluded. This confirms the original cluster was a noise artifact of small sample CTR, not a real content archetype.

Population selection: The clustering only includes pages with gsc_avg_position > 0 pages that already have search visibility. In the March data, 279,294 distinct pages were excluded under this filter for having no recorded ranking position at all, compared to 175,304 pages that were included. This is a disclosed and reasonable scoping choice (unranked pages don't have the position/CTR signal the clustering needs), but it means the clusters describe visible content only, and any claims from this model should not be extended to invisible or unindexed pages.

No future window leakage was found: all features are computed within a single month's aggregate, with no forward looking data folded into any feature.

In [6]:
X_no_ctr = features_df[["avg_position", "impressions_total", "word_count"]]
X_no_ctr_scaled = StandardScaler().fit_transform(X_no_ctr)
km_no_ctr = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_no_ctr_scaled)

X_scaled_march_full = StandardScaler().fit_transform(features_df[["avg_position", "impressions_total", "ctr", "word_count"]])
km_march_full = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_scaled_march_full)

sil_with_ctr = silhouette_score(X_scaled_march_full, km_march_full.labels_, sample_size=10000, random_state=42)
sil_without_ctr = silhouette_score(X_no_ctr_scaled, km_no_ctr.labels_, sample_size=10000, random_state=42)
print(f"Silhouette WITH ctr: {sil_with_ctr:.3f}")
print(f"Silhouette WITHOUT ctr: {sil_without_ctr:.3f}")

no_ctr_centers = pd.DataFrame(km_no_ctr.cluster_centers_, columns=["avg_position","impressions_total","word_count"])
print(no_ctr_centers.round(3))

Silhouette WITH ctr: 0.474
Silhouette WITHOUT ctr: 0.496
   avg_position  impressions_total  word_count
0        -0.387             -0.069      -0.250
1         1.990             -0.232      -0.094
2        -0.072             -0.005       2.115
3        -0.326              5.766       0.201


In [7]:
filtered_df = features_df[features_df["impressions_total"] >= 20].copy()
X_filtered = filtered_df[["avg_position", "impressions_total", "ctr", "word_count"]]
X_filtered_scaled = StandardScaler().fit_transform(X_filtered)

km_filtered = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_filtered_scaled)
filtered_df["cluster"] = km_filtered.labels_
filtered_profile = filtered_df.groupby("cluster")[["avg_position","impressions_total","ctr","word_count"]].mean()
filtered_profile["n_pages"] = filtered_df["cluster"].value_counts().sort_index()
print(filtered_profile.round(3))
print("\nRows dropped by filter:", len(features_df) - len(filtered_df))

         avg_position  impressions_total    ctr  word_count  n_pages
cluster                                                             
0              17.596           7916.741  0.002     5009.92    10843
1              48.981            511.858  0.001    2700.867    21063
2              11.532            519.933  0.031    2612.465     3404
3              10.054           1903.939  0.002    2682.765    95344

Rows dropped by filter: 44650


In [9]:
excluded = con.execute("""
    SELECT COUNT(DISTINCT content_hash_id) as n_excluded
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_avg_position <= 0 OR gsc_avg_position IS NULL
""").df()
print(excluded)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   n_excluded
0      279294


## 4. Claim rewrite
Original (Week 5): "Cluster 2: 'High-Traffic Workhorses' massive impressions (32,220 avg), decent position (11.6), typical CTR. These are the site's real traffic drivers protect, don't disturb."

This overclaims in two ways: it treats one static month of cluster membership as proof of causal importance, and "protect, don't disturb" is a prescriptive instruction with no evidence behind it the clustering has no time dimension, so there's no basis to claim that changing these pages would cause harm.

Rewritten: "Cluster 2 pages show high impressions and a workable average position in the observed month. The out of time check (Feb fit applied to March) shows this cluster reappears with a similar profile, which is directionally consistent with it being a stable, high-visibility group rather than a one-month fluke. This is a decision-support signal to prioritize monitoring these pages before making changes not evidence that changes to them would be harmful, since no causal or before/after test was run on actual edits."

In [10]:
print(oot_profile.round(3))

             avg_position  impressions_total    ctr  word_count  n_pages
cluster_oot                                                             
0                  14.031           7161.803  0.003    4191.313    23393
1                  51.125            276.307  0.001     2643.38    29462
2                   9.431            855.552  0.004    2485.221   122157
3                   8.714              1.932  0.727    1423.274      292


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.